In [16]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

In [18]:
import  torch.nn.functional as F

In [20]:
import re
import unicodedata

In [46]:
import pickle
import numpy as np

In [22]:
def clean_text(text):
    # Normalize unicode
    text = unicodedata.normalize("NFKD", text)
    
    # Remove HTML tags like <br/>
    text = re.sub(r'<.*?>', ' ', text)
    
    # Remove URLs
    text = re.sub(r'http\S+|www.\S+', '', text)
    
    # Remove non-printable characters
    text = ''.join(ch for ch in text if ch.isprintable())
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [72]:
with open("label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

In [24]:
model_path = "covid-sentiment-bert-model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

In [26]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [28]:
def predict(text):
    tokens = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

    with torch.no_grad():
        outputs = model(**tokens)
        print(outputs)
        print(outputs.logits) # Logits
        print(F.softmax(outputs.logits)) # Probabilities
        predicted_class = torch.argmax(outputs.logits, dim=-1).item()

    return predicted_class

In [62]:
# text = "TRENDING: New Yorkers encounter empty supermarket shelves (pictured, Wegmans in Brooklyn), sold-out online grocers (FoodKick, axDelivery) as #coronavirus-fearing shoppers stock up https://t.co/Gr76pcrLWh https://t.co/ivMKMsqdT1"
text = 'Find out how you can protect yourself and loved ones from #coronavirus. ?'

In [78]:
cleaned_text = clean_text(text)
cleaned_text

'Find out how you can protect yourself and loved ones from #coronavirus. ?'

In [80]:
predicted_class = predict(text)

SequenceClassifierOutput(loss=None, logits=tensor([[-1.3626,  4.9646, -1.7266, -1.8793,  0.5209]], device='mps:0'), hidden_states=None, attentions=None)
tensor([[-1.3626,  4.9646, -1.7266, -1.8793,  0.5209]], device='mps:0')
tensor([[0.0018, 0.9844, 0.0012, 0.0010, 0.0116]], device='mps:0')


/var/folders/gc/npgf58h95t16nm6m6d8q9pcr0000gn/T/ipykernel_1356/759220889.py:8: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  print(F.softmax(outputs.logits)) # Probabilities


In [82]:
pred = []
pred.append(predicted_class)

In [84]:
np.array(pred)

array([1])

In [87]:
final_pred = le.inverse_transform(np.array(pred))[0]

In [89]:
final_pred

'Extremely Positive'